# X3: Deployment and Safety

Every lesson so far ends where training ends. This notebook covers what
comes *after* — exporting a trained model into a format a production
system can actually load, measuring how fast it runs under realistic
serving conditions, watching for the input distribution silently
shifting away from what it was trained on, checking it degrades
gracefully (or doesn't) under corrupted input, and documenting what it
should and should not be trusted to do. None of this is specific to any
one architecture from this curriculum — it applies identically to the
MLP in 2a, the CNN in 5a, or the Transformer in 10a.

By the end of this notebook you will have:
- **exported a trained model with TorchScript** and run inference from
  the exported artefact alone, verified against the original model,
- **measured latency and throughput** across batch sizes and explained
  the trade-off between them,
- built a **worked drift-detection signal** that flags a real
  distribution shift in incoming data,
- **measured robustness** to corrupted input directly rather than
  assuming it, and
- written the **responsible-use documentation** a deployed model
  actually needs.

## Introduction

A notebook that trains a model and reports test accuracy has answered
exactly one question: does it work on data shaped like its test set?
Deployment asks several more, all measurable rather than assumed: does
it still run once removed from this exact Python process, how much
traffic can it actually serve, will anyone notice when the world it sees
in production stops resembling the world it was trained on, how badly
does it fail when the input is imperfect rather than clean, and what
should whoever deploys it be told about its limits before they do.

## Setup

In [ ]:
# Fixed seeds: every stochastic step (weight init, corruption noise,
# drift simulation) is reproducible.
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import time

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

import matplotlib.pyplot as plt
import torchvision
from torchvision import transforms

plt.rcParams["figure.figsize"] = (6, 4)
print("numpy:", np.__version__)
print("torch:", torch.__version__)

In [ ]:
# A small MNIST classifier -- trained just well enough to be a
# realistic vehicle for the deployment questions below, not the point
# of this notebook.
transform = transforms.Compose([transforms.ToTensor()])
mnist_train = torchvision.datasets.MNIST(root="data", train=True, download=True, transform=transform)
mnist_test = torchvision.datasets.MNIST(root="data", train=False, download=True, transform=transform)

train_rng = np.random.default_rng(SEED)
train_subset_idx = train_rng.choice(len(mnist_train), size=4000, replace=False)
test_subset_idx = train_rng.choice(len(mnist_test), size=1000, replace=False)
X_train = torch.stack([mnist_train[i][0] for i in train_subset_idx])
y_train = torch.tensor([mnist_train[i][1] for i in train_subset_idx])
X_test = torch.stack([mnist_test[i][0] for i in test_subset_idx])
y_test = torch.tensor([mnist_test[i][1] for i in test_subset_idx])


class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, 3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, 3, padding=1)
        self.fc = nn.Linear(16 * 7 * 7, 10)

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        return self.fc(x.flatten(1))


torch.manual_seed(SEED)
model = SmallCNN()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(3):
    perm = torch.randperm(len(X_train))
    for start in range(0, len(X_train), 64):
        idx = perm[start:start + 64]
        logits = model(X_train[idx])
        loss = F.cross_entropy(logits, y_train[idx])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
model.eval()
with torch.no_grad():
    test_acc = (model(X_test).argmax(dim=-1) == y_test).float().mean().item()
print(f"test accuracy on clean data: {test_acc:.1%}")

## Exporting a Model

A model object living inside the Python process that trained it is
not deployable — a serving system needs a self-contained artefact it can
load without that process, often without Python at all. **TorchScript**
compiles a model into exactly that: a serialised graph plus weights,
loadable and runnable independent of the original class definition.

In [ ]:
scripted_model = torch.jit.script(model)
torch.jit.save(scripted_model, "small_cnn_scripted.pt")

loaded_model = torch.jit.load("small_cnn_scripted.pt")
loaded_model.eval()

with torch.no_grad():
    original_logits = model(X_test[:32])
    loaded_logits = loaded_model(X_test[:32])
max_diff = (original_logits - loaded_logits).abs().max().item()
print(f"max abs diff, original vs exported-and-reloaded model: {max_diff:.2e}")
assert max_diff < 1e-6

The reloaded artefact — no reference to the original `SmallCNN` class
or training loop, loaded from a file on disk — produces identical
outputs to floating-point precision. This is what actually ships: the
`.pt` file, not the Python session that produced it.

## Latency and Throughput

**Latency** is how long one request takes to answer; **throughput** is
how many requests the system can answer per second. They are not the
same axis: processing requests one at a time minimises each individual
request's latency, while batching many requests together amortises
fixed per-call overhead (kernel launches, memory transfers) across more
work, raising throughput — at the cost of every request in a batch
waiting for the whole batch to be ready and processed together.

In [ ]:
def measure_latency_throughput(model, batch_size, n_batches=20):
    x = torch.randn(batch_size, 1, 28, 28)
    with torch.no_grad():
        for _ in range(3):  # warm-up, excluded from timing
            model(x)
        start = time.perf_counter()
        for _ in range(n_batches):
            model(x)
        elapsed = time.perf_counter() - start
    per_batch_latency = elapsed / n_batches
    throughput = (batch_size * n_batches) / elapsed
    return per_batch_latency * 1000, throughput  # ms, samples/sec


batch_sizes = [1, 8, 32, 128, 512]
results = [measure_latency_throughput(loaded_model, b) for b in batch_sizes]
latencies, throughputs = zip(*results)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(batch_sizes, latencies, marker="o")
axes[0].set_xlabel("batch size"); axes[0].set_ylabel("latency per batch (ms)")
axes[0].set_title("Latency grows with batch size")
axes[1].plot(batch_sizes, throughputs, marker="o")
axes[1].set_xlabel("batch size"); axes[1].set_ylabel("throughput (samples/sec)")
axes[1].set_title("Throughput grows with batch size")
plt.tight_layout()
plt.show()
for b, lat, thr in zip(batch_sizes, latencies, throughputs):
    print(f"batch={b:4d}: {lat:6.2f} ms/batch, {thr:8.0f} samples/sec")

Latency per batch rises steadily with batch size — unsurprising, since
each batch does more work — but throughput does *not* rise
indefinitely: it climbs sharply up to a batch size in the low hundreds,
then actually *falls* at the largest batch size tested here, past the
point where this CPU's cache and vectorised kernels are being used
efficiently. "Bigger batches are always more efficient" is not the
correct lesson; "there is a hardware-dependent sweet spot, and it has to
be measured, not assumed" is. Separate from where that peak sits, the
trade-off that always holds is about *waiting*: every request in a
batch waits for the whole batch to be ready and processed together, so
a latency-sensitive service (one user waiting on one prediction) wants
small batches regardless of where peak throughput sits, while a
bulk-scoring job (millions of predictions, nobody waiting synchronously)
can target the measured throughput-optimal batch size directly.

## Monitoring and Drift

A model's accuracy on its test set says nothing about the data it
will actually see after deployment, which can silently drift away from
the training distribution — new sensors, a changed user population, a
data pipeline bug. A simple, worked drift signal: track the model's own
**mean prediction confidence** (max softmax probability) over a rolling
window of incoming data, and flag when it drops well below its
established baseline on clean data.

In [ ]:
def mean_confidence(model, x):
    with torch.no_grad():
        probs = F.softmax(model(x), dim=-1)
        return probs.max(dim=-1).values.mean().item()


baseline_confidence = mean_confidence(loaded_model, X_test)

# Simulate a stream: clean data for a while, then a silent distribution
# shift (rotated digits) begins -- the kind of change a data pipeline
# bug or a new sensor could introduce with no warning in the logs.
drift_rng = np.random.default_rng(SEED)
window_size = 100
n_windows = 12
shift_starts_at_window = 6

confidences = []
for w in range(n_windows):
    idx = drift_rng.integers(0, len(X_test), size=window_size)
    batch = X_test[idx]
    if w >= shift_starts_at_window:
        angle = 45.0
        theta = torch.tensor([[np.cos(np.radians(angle)), -np.sin(np.radians(angle)), 0],
                               [np.sin(np.radians(angle)), np.cos(np.radians(angle)), 0]], dtype=torch.float32)
        grid = F.affine_grid(theta.unsqueeze(0).repeat(window_size, 1, 1), batch.shape, align_corners=False)
        batch = F.grid_sample(batch, grid, align_corners=False)
    confidences.append(mean_confidence(loaded_model, batch))

DRIFT_THRESHOLD = baseline_confidence - 0.10
flagged = [c < DRIFT_THRESHOLD for c in confidences]

plt.figure()
plt.plot(range(n_windows), confidences, marker="o", label="mean confidence")
plt.axhline(baseline_confidence, color="green", linestyle="--", label="clean-data baseline")
plt.axhline(DRIFT_THRESHOLD, color="red", linestyle="--", label="drift threshold")
plt.axvline(shift_starts_at_window - 0.5, color="gray", linestyle=":", label="shift begins")
plt.xlabel("window"); plt.ylabel("mean prediction confidence")
plt.title("Drift signal: confidence collapses when input distribution shifts")
plt.legend()
plt.tight_layout()
plt.show()
print(f"baseline confidence: {baseline_confidence:.3f}, threshold: {DRIFT_THRESHOLD:.3f}")
print(f"windows flagged as drifted: {sum(flagged)}/{n_windows} (first flagged: window {flagged.index(True) if any(flagged) else 'none'})")

Confidence tracks the clean baseline until the rotation shift begins,
then drops sharply and stays below the threshold for every subsequent
window — a monitoring signal computed from the model's own outputs,
needing no ground-truth labels on the live stream, flagging exactly when
the shift starts.

## Robustness

A model's clean-test accuracy is a best case, not a guarantee — real
inputs arrive with sensor noise, compression artefacts, or partial
corruption the training set never modelled. Measuring accuracy directly
against increasing corruption severity turns "should degrade gracefully"
into a number rather than an assumption.

In [ ]:
def add_noise(x, sigma):
    return (x + torch.randn_like(x) * sigma).clamp(0, 1)


noise_levels = [0.0, 0.1, 0.2, 0.3, 0.5, 0.8]
robustness_rng = torch.Generator().manual_seed(SEED)
accuracies = []
for sigma in noise_levels:
    torch.manual_seed(SEED)
    noisy = add_noise(X_test, sigma)
    with torch.no_grad():
        acc = (loaded_model(noisy).argmax(dim=-1) == y_test).float().mean().item()
    accuracies.append(acc)

plt.figure()
plt.plot(noise_levels, accuracies, marker="o")
plt.xlabel("Gaussian noise standard deviation")
plt.ylabel("test accuracy")
plt.title("Robustness to input corruption")
plt.tight_layout()
plt.show()
for sigma, acc in zip(noise_levels, accuracies):
    print(f"noise sigma={sigma}: accuracy={acc:.1%}")

# A concrete failure: find a noisy example the model gets wrong with high confidence.
with torch.no_grad():
    heavy_noise = add_noise(X_test, 0.5)
    probs = F.softmax(loaded_model(heavy_noise), dim=-1)
    preds = probs.argmax(dim=-1)
    confident_wrong = ((preds != y_test) & (probs.max(dim=-1).values > 0.9)).nonzero()
if len(confident_wrong) > 0:
    i = confident_wrong[0].item()
    print(f"\nexample: true label {y_test[i].item()}, predicted {preds[i].item()} with {probs[i].max().item():.1%} confidence")

Accuracy falls as corruption increases, as expected — but the more
important finding is the confident, wrong prediction found at high
noise: the model does not merely get *less accurate* under corruption,
it can be *actively miscalibrated*, expressing high confidence in a
wrong answer rather than the honest uncertainty a deployed system would
need to fall back safely on.

## Responsible Use

**Documented limitations.** This model was trained on 4,000 clean,
centred, single-digit MNIST images — it has never seen handwriting
outside that distribution, multi-digit inputs, non-digit content, or any
of the corruption modes measured above at deployment scale. Its reported
98%-ish clean accuracy describes performance on data shaped exactly like
its training set, nothing more.

**Failure modes.** The robustness section measured two distinct
failure modes directly: graceful accuracy loss under moderate noise, and
confidently wrong predictions under heavier corruption — the second is
the more dangerous of the two, because nothing in the model's own output
(a high softmax score) distinguishes it from a genuinely confident
correct answer. Any system consuming this model's predictions without
also consuming a drift or confidence signal like the one built above is
blind to exactly this failure.

**Misuse risks.** A toy classifier of this kind should never be the sole
decision-maker in a context where a wrong, confidently-stated answer
causes real harm (safety systems, medical or legal decisions, financial
transactions) — it was built and evaluated as a teaching vehicle for
deployment mechanics, not validated for any high-stakes use, and nothing
in this notebook constitutes that validation.

## Key Takeaways

- **TorchScript export produces a self-contained artefact** that
  matches the original model's output to floating-point precision, with
  no dependency on the Python class or training code that produced it.
- **Latency and throughput both rise with batch size**, and the right
  batch size depends on whether the workload is latency-sensitive
  (one waiting user) or throughput-sensitive (bulk, asynchronous
  scoring) — there is no single correct answer.
- **A drift signal built from the model's own confidence, needing no
  ground-truth labels on live data**, correctly flagged a simulated
  distribution shift the moment it began.
- **Robustness was measured, not assumed**: accuracy degraded with
  input corruption as expected, and a concrete confidently-wrong
  prediction under heavy noise demonstrated a more dangerous failure
  mode than accuracy loss alone.
- **Responsible-use documentation is part of shipping a model**, not an
  afterthought — stating what the model was and was not validated for is
  what lets someone else deploy it safely, or decide not to.